# Práctica 7: Seguimiento de objetos
Vas a usar un tracker distinto al de la lección. Si te trabás, repasá la lección en `../notebooks/7_seguimiento_objetos.ipynb`.

### Ejercicio 1: Probar TrackerMOSSE en lugar de CSRT
La lección dejó comentado `TrackerMOSSE_create()`. Usalo en este ejercicio y compará su comportamiento con CSRT.

In [ ]:
import cv2  # Qué: importa OpenCV. Cómo: expone cv2.legacy con los trackers clásicos y las utilidades de captura/GUI. Por qué: es la única dependencia necesaria para el seguimiento de objetos.

# TODO: creá el tracker con cv2.legacy.TrackerMOSSE_create() en vez de
# TrackerCSRT_create()
# Pista: MOSSE es un filtro de correlación mucho más liviano que CSRT (más rápido, pero típicamente menos robusto ante cambios de escala u oclusión); fijate cómo se creó el tracker CSRT en la lección (notebooks/7_seguimiento_objetos.ipynb) y la línea comentada de referencia a MOSSE que dejó esa misma lección.
tracker = None

cap = cv2.VideoCapture(0)  # Qué: abre la webcam por defecto (índice 0). Por qué: fuente de video sobre la que se va a seguir el objeto elegido.

ret, frame = cap.read()  # Qué: captura un único cuadro inicial. Cómo: `ret` indica éxito, `frame` es la imagen BGR. Por qué: se necesita un cuadro estático para que el usuario seleccione manualmente el objeto a rastrear.

bbox = cv2.selectROI("Selecciona el objeto", frame, False)  # Qué: abre una ventana interactiva para dibujar con el mouse el rectángulo (ROI) del objeto a seguir. Cómo: devuelve la tupla (x, y, w, h); `False` desactiva el crosshair centrado. Por qué: el tracker necesita una caja delimitadora inicial para aprender la apariencia del objeto.
cv2.destroyWindow("Selecciona el objeto")  # Qué: cierra específicamente la ventana de selección. Por qué: ya cumplió su propósito y no debe seguir ocupando pantalla durante el seguimiento.

# TODO: inicializá el tracker con tracker.init(frame, bbox)
# Pista: es el paso que le "enseña" al tracker cómo es el objeto a partir del cuadro inicial y la bbox seleccionada; sin esto no se puede llamar a tracker.update() más abajo.
ok = None

while True:  # Qué: procesa la webcam en vivo, cuadro por cuadro, actualizando el seguimiento hasta que el usuario salga.
    ret, frame = cap.read()  # Qué: captura el siguiente cuadro del video. Cómo: `ret` indica éxito, `frame` es la imagen BGR actual.

    # TODO: actualizá el tracker con tracker.update(frame)
    # Pista: en cada cuadro nuevo el tracker debe recalcular dónde se movió el objeto respecto al cuadro anterior; el resultado son dos valores: si logró seguirlo y la nueva posición/tamaño del objeto.
    success, bbox = None, None

    if success:  # Qué: rama que se ejecuta cuando el tracker sigue confiando en su seguimiento.
        x, y, w, h = map(int, bbox)  # Qué: descompone bbox en sus 4 componentes y los convierte a enteros. Por qué: las funciones de dibujo de OpenCV requieren coordenadas enteras en píxeles.
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)  # Qué: dibuja un rectángulo verde alrededor del objeto rastreado (color distinto al azul de CSRT en la lección, para diferenciar visualmente que acá se usa MOSSE). Cómo: traza desde (x,y) hasta (x+w, y+h) con grosor 2.
        cv2.putText(frame, "MOSSE", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)  # Qué: escribe la etiqueta "MOSSE" arriba del rectángulo. Por qué: identifica en pantalla qué algoritmo de tracking se está usando en este ejercicio.
    else:  # Qué: rama que se ejecuta cuando el tracker perdió el objeto.
        cv2.putText(frame, "Perdido", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)  # Qué: muestra "Perdido" en rojo en posición fija. Por qué: informa que el seguimiento falló en vez de dibujar una bbox inválida.

    cv2.imshow("Seguimiento con MOSSE", frame)  # Qué: muestra el cuadro actual con las anotaciones dibujadas. Por qué: salida visual en vivo del seguimiento.

    if cv2.waitKey(30) == ord('q'):  # Qué: espera 30 ms por una tecla y compara con 'q'. Por qué: regula la tasa de refresco del loop y permite salir de forma controlada.
        break  # Qué: corta el bucle infinito. Por qué: única salida controlada del loop.

cap.release()  # Qué: libera el dispositivo de cámara. Por qué: evita bloquear la webcam para otros procesos.
cv2.destroyAllWindows()  # Qué: cierra todas las ventanas abiertas de OpenCV. Por qué: limpieza de recursos de GUI al terminar.

# TODO: agregá un comentario con tus observaciones sobre velocidad y precisión
# de MOSSE frente a CSRT
# Pista: pensá en qué pasa cuando el objeto se mueve rápido, cambia de tamaño (se acerca/aleja de cámara), o queda parcialmente tapado, y comparalo con lo que viste al usar CSRT en la lección.